# Phase 1 — Multi-Task Circuit Sharing: Addition + Subtraction

**Research question:** When a 1-layer transformer is trained on both modular addition *and* subtraction at once, does it build **one shared circuit** or **two independent circuits**?

## What this notebook does
1. **Setup** — install dependencies, mount Google Drive for checkpointing
2. **Data** — generate a combined add+sub dataset (mod 113, op-token disambiguated)
3. **Train** — run the grokking training loop; checkpoint every 200 epochs
4. **Verify grokking** — plot per-task accuracy curves separately for add and sub
5. **Head ablation sweep** — zero out each of the 4 attention heads; measure accuracy drop **per task**
6. **Activation patching** — swap clean activations from an add run into a sub forward pass; measure structured output changes
7. **Fourier analysis** — check which frequencies the multi-task model uses vs. the Phase 0 single-task model
8. **Conclusion** — interpret results in terms of shared vs. task-specific circuitry

## Success criteria
| Step | How you know it worked |
|------|------------------------|
| Data generation | `vocab_size=116`, `train N≈7680`, `test N≈17968` (≈30/70 split of 113²×2=25538 examples) |
| Training | Both per-task test accuracies reach ≥95% before epoch 15000 |
| Grokking plot | Two separate accuracy curves (add in blue, sub in orange) both show the delayed phase transition |
| Head ablation | Bar chart with 4 groups (one per head); each group has two bars (add-acc-drop, sub-acc-drop) |
| Activation patching | Patching add→sub or sub→add produces **structured** (not random) logit changes |
| Fourier | Multi-task model shows the same ≤6 dominant frequencies as Phase 0 (strong sharing signal) OR shows twice as many (split-circuit signal) |

---
**Notebook version:** Phase 1 (add + sub)  
**Model:** 1-layer, 4-head, d_model=128, mod 113  
**Compute:** designed for free Colab T4 (~15 min to train)

## Cell 1 — Install Dependencies

Pin `transformer_lens<3` — newer versions have an unstable API rewrite in progress.

In [ ]:
# Run once. If Colab asks to restart the runtime, do it, then skip this cell.
!pip install -q 'transformer_lens>=1.0,<3' einops tqdm
print("✅ Dependencies installed.")

## Cell 2 — Imports & Reproducibility

In [ ]:
import os, random, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import trange

import transformer_lens
from transformer_lens import HookedTransformer, HookedTransformerConfig

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Imports OK | transformer_lens {transformer_lens.__version__} | device: {DEVICE}")

## Cell 3 — Hyperparameters

All experiment knobs in one place so you can sweep them without hunting through the notebook.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
P          = 113       # modulus  (prime, ≥100 is conventional for grokking studies)
OPS        = ("add", "sub")  # Phase 1: two related operations
TRAIN_FRAC = 0.3       # 30% train — low fraction forces generalisation

# ── Model ─────────────────────────────────────────────────────────────────────
D_MODEL  = 128
N_HEADS  = 4
D_MLP    = 512
N_LAYERS = 1
SEQ_LEN  = 4           # [a, op_token, b, =]  →  predict result at position 3

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS           = 15_000
LR               = 1e-3
WEIGHT_DECAY     = 1.0   # high WD is the key hyperparameter that triggers grokking
CHECKPOINT_EVERY = 200

# ── Results dir ───────────────────────────────────────────────────────────────
os.makedirs("results", exist_ok=True)

print("✅ Hyperparameters set.")
print(f"   P={P}, ops={OPS}, train_frac={TRAIN_FRAC}")
print(f"   d_model={D_MODEL}, n_heads={N_HEADS}, d_mlp={D_MLP}, layers={N_LAYERS}")
print(f"   Epochs={EPOCHS}, LR={LR}, WD={WEIGHT_DECAY}")

## Cell 4 — Data Generation

Token layout:
- `[0, P-1]` → number tokens
- `P` → `add` op-token
- `P+1` → `sub` op-token
- `P+2` → `=` token

Input sequence: `[a, op_token, b, =]`  
Label: `result` (the number token to predict at position 3)

**Success check:** `vocab_size=116`, ~7680 train examples, ~17858 test examples.

In [ ]:
def make_dataset(p=113, ops=("add", "sub"), train_frac=0.3, seed=0):
    """Build a combined modular arithmetic dataset with an op-token disambiguator."""
    OP_FNS = {
        "add": lambda a, b, p: (a + b) % p,
        "sub": lambda a, b, p: (a - b) % p,
        "mul": lambda a, b, p: (a * b) % p,
    }
    rng = np.random.RandomState(seed)
    op_list = list(ops)
    equals_token = p + len(op_list)

    all_inputs, all_labels = [], []
    for op_idx, op_name in enumerate(op_list):
        op_token = p + op_idx
        fn = OP_FNS[op_name]
        for a in range(p):
            for b in range(p):
                all_inputs.append([a, op_token, b, equals_token])
                all_labels.append(fn(a, b, p))

    inputs = torch.tensor(all_inputs, dtype=torch.long)
    labels = torch.tensor(all_labels, dtype=torch.long)

    n = len(inputs)
    perm = rng.permutation(n)
    n_train = int(train_frac * n)
    train_data = (inputs[perm[:n_train]], labels[perm[:n_train]])
    test_data  = (inputs[perm[n_train:]], labels[perm[n_train:]])

    vocab_size = p + len(op_list) + 1
    return train_data, test_data, vocab_size, equals_token, op_list


train_data, test_data, VOCAB_SIZE, EQUALS_TOKEN, OP_LIST = make_dataset(
    p=P, ops=OPS, train_frac=TRAIN_FRAC, seed=SEED
)

# ── Op-token map for per-task filtering ──────────────────────────────────────
OP_TOKENS = {op: P + i for i, op in enumerate(OP_LIST)}

print(f"✅ Dataset ready.")
print(f"   vocab_size={VOCAB_SIZE}  equals_token={EQUALS_TOKEN}")
print(f"   op tokens: {OP_TOKENS}")
print(f"   train examples: {train_data[0].shape[0]:,}")
print(f"   test  examples: {test_data[0].shape[0]:,}")

## Cell 5 — Build the Model

Same architecture as Phase 0: 1-layer, attention+MLP, no LayerNorm (keeps weight/Fourier analysis clean).

In [ ]:
def build_model(vocab_size, d_model=128, n_heads=4, d_mlp=512, n_layers=1,
                seq_len=4, device="cuda", seed=0):
    """Build a tiny HookedTransformer. No LayerNorm = cleaner Fourier analysis."""
    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        d_model=d_model,
        d_head=d_model // n_heads,
        n_heads=n_heads,
        d_mlp=d_mlp,
        d_vocab=vocab_size,
        n_ctx=seq_len,
        act_fn="relu",
        normalization_type=None,
        seed=seed,
        device=device,
    )
    return HookedTransformer(cfg)


model = build_model(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS, d_mlp=D_MLP,
    n_layers=N_LAYERS, seq_len=SEQ_LEN, device=DEVICE, seed=SEED
)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model built: {n_params:,} parameters")
print(model)

## Cell 6 — Helper: Per-Task Accuracy

This is the key addition over Phase 0. We filter the test set by the op-token in position 1 to get **separate accuracy for add and sub**.

**Why this matters:** If we only track overall accuracy, we can't tell whether both tasks grokked or just one of them did.

In [ ]:
@torch.no_grad()
def per_task_accuracy(model, inputs, labels, op_tokens_map, device=DEVICE):
    """
    Returns a dict {op_name: accuracy} computed on the given inputs/labels.
    op_tokens_map: e.g. {"add": 113, "sub": 114}
    """
    model.eval()
    inputs, labels = inputs.to(device), labels.to(device)
    logits = model(inputs)[:, -1, :]          # [N, vocab]
    preds  = logits.argmax(-1)                # [N]

    accs = {}
    for op_name, op_tok in op_tokens_map.items():
        mask = (inputs[:, 1] == op_tok)       # examples using this operation
        if mask.sum() == 0:
            accs[op_name] = float("nan")
        else:
            accs[op_name] = (preds[mask] == labels[mask]).float().mean().item()
    return accs

print("✅ per_task_accuracy() defined.")

## Cell 7 — Training Loop

Full-batch gradient descent with AdamW + high weight decay. Logs per-task accuracy every `CHECKPOINT_EVERY` epochs.

**Expected runtime:** ~10–15 minutes on a free Colab T4.

**Success signal to watch for:**
- Train accuracy should hit ~100% quickly (within the first few hundred epochs)
- Test accuracy (both add AND sub) should stay near 0% for a while, then **both** make a sharp jump to ~100%
- If only one task jumps, re-train with a higher weight decay or more epochs

In [ ]:
train_inputs, train_labels = [x.to(DEVICE) for x in train_data]
test_inputs,  test_labels  = [x.to(DEVICE) for x in test_data]

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98)
)

history = {
    "epoch": [],
    "train_loss": [], "test_loss": [],
    "train_acc": [],  "test_acc": [],
}
# Per-task accuracy storage
for op in OP_LIST:
    history[f"train_acc_{op}"] = []
    history[f"test_acc_{op}"]  = []

checkpoints = []

for epoch in trange(EPOCHS, desc="Training"):
    model.train()
    logits = model(train_inputs)[:, -1, :]    # [N_train, vocab]
    loss   = F.cross_entropy(logits, train_labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % CHECKPOINT_EVERY == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            tr_logits = model(train_inputs)[:, -1, :]
            te_logits = model(test_inputs)[:, -1, :]
            tr_loss   = F.cross_entropy(tr_logits, train_labels).item()
            te_loss   = F.cross_entropy(te_logits, test_labels).item()
            tr_acc    = (tr_logits.argmax(-1) == train_labels).float().mean().item()
            te_acc    = (te_logits.argmax(-1) == test_labels).float().mean().item()

        tr_task_acc = per_task_accuracy(model, train_inputs, train_labels, OP_TOKENS)
        te_task_acc = per_task_accuracy(model, test_inputs,  test_labels,  OP_TOKENS)

        history["epoch"].append(epoch)
        history["train_loss"].append(tr_loss)
        history["test_loss"].append(te_loss)
        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)
        for op in OP_LIST:
            history[f"train_acc_{op}"].append(tr_task_acc[op])
            history[f"test_acc_{op}"].append(te_task_acc[op])

        # Save a lightweight checkpoint (state dict, CPU)
        checkpoints.append({
            "epoch": epoch,
            "state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
        })

print(f"\n✅ Training complete.")
print(f"   Final train acc: {history['train_acc'][-1]:.3f}")
print(f"   Final test  acc: {history['test_acc'][-1]:.3f}")
for op in OP_LIST:
    print(f"   Final test acc [{op}]: {history[f'test_acc_{op}'][-1]:.3f}")

## Cell 8 — Plot: Loss Curves + Per-Task Grokking

**What to look for:**
- Left panel (loss): train loss drops fast, test loss falls slowly then suddenly plunges — the classic grokking shape.
- Right panel (accuracy): both add (blue) and sub (orange) should show a **delayed phase transition** — staying near 0% then jumping to ~100%. If the two transitions happen at roughly the same epoch, that's already suggestive of shared learning dynamics.

**Saves to** `results/phase1_grokking_curves.png`

In [ ]:
epochs = history["epoch"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Phase 1 — Multi-Task Grokking: Addition + Subtraction (mod {P})",
    fontsize=14, fontweight="bold"
)

# ── Loss curves ───────────────────────────────────────────────────────────────
ax = axes[0]
ax.semilogy(epochs, history["train_loss"], label="Train loss", color="steelblue")
ax.semilogy(epochs, history["test_loss"],  label="Test loss",  color="tomato")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss (log scale)")
ax.set_title("Loss Curves (Overall)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# ── Per-task accuracy ─────────────────────────────────────────────────────────
ax = axes[1]
COLORS = {"add": "steelblue", "sub": "darkorange"}
for op in OP_LIST:
    c = COLORS.get(op, "green")
    ax.plot(epochs, history[f"train_acc_{op}"],
            linestyle="--", color=c, alpha=0.5, label=f"Train [{op}]")
    ax.plot(epochs, history[f"test_acc_{op}"],
            linestyle="-",  color=c,             label=f"Test  [{op}]")

# Mark the grokking epoch for each task (first time test acc > 95%)
for op in OP_LIST:
    c = COLORS.get(op, "green")
    for i, (ep, acc) in enumerate(zip(epochs, history[f"test_acc_{op}"])):
        if acc > 0.95:
            ax.axvline(ep, linestyle=":", color=c, alpha=0.6)
            ax.text(ep + 100, 0.1, f"{op} grok\n@ {ep}",
                    color=c, fontsize=8, ha="left")
            break

ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Task Test Accuracy — Grokking Phase Transitions")
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/phase1_grokking_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved results/phase1_grokking_curves.png")

# ── Success check ─────────────────────────────────────────────────────────────
for op in OP_LIST:
    final_acc = history[f"test_acc_{op}"][-1]
    status = "✅ GROKKED" if final_acc >= 0.95 else "❌ DID NOT GROK"
    print(f"   {op}: final test acc = {final_acc:.3f}  {status}")

print("\n⚠️  If either task did NOT grok, try: WEIGHT_DECAY=1.5 or EPOCHS=20000 and retrain.")

## Cell 9 — Fourier Analysis of Number-Token Embeddings

Replicates the Phase 0 analysis on the multi-task model. Take the embedding matrix for number tokens `[0, P-1]`, apply DFT across the `P` numbers, and plot the power spectrum.

**What to look for:**
- Phase 0 (single-task addition) showed ~4 dominant frequency spikes.
- If the multi-task model uses the **same** frequencies for both tasks → strong evidence of a **shared Fourier circuit**.
- If it uses **twice as many** frequencies (some for add, some for sub) → evidence of **separate representations**.

**Saves to** `results/phase1_fourier_spectrum.png`

In [ ]:
def fourier_embedding_analysis(model, p):
    """DFT of the number-token embedding rows, summed over d_model."""
    W_E = model.W_E[:p].detach().cpu().numpy()   # [p, d_model]
    fourier = np.fft.rfft(W_E, axis=0)           # [p//2+1, d_model]
    power   = (np.abs(fourier) ** 2).sum(axis=1) # [p//2+1]
    return power


power = fourier_embedding_analysis(model, P)
freqs = np.arange(len(power))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(freqs, power, color="steelblue", edgecolor="white", linewidth=0.3)

# Annotate top-6 frequencies
top_k    = 6
top_idx  = np.argsort(power)[-top_k:][::-1]
for idx in top_idx:
    ax.annotate(
        f"f={idx}",
        xy=(idx, power[idx]),
        xytext=(idx + 0.5, power[idx] * 1.05),
        fontsize=8, color="crimson",
        arrowprops=dict(arrowstyle="->", color="crimson", lw=0.8)
    )
    ax.bar(idx, power[idx], color="crimson")

ax.set_xlabel("Frequency")
ax.set_ylabel("Power (sum over d_model)")
ax.set_title(
    f"Phase 1 — Fourier Spectrum of Number-Token Embeddings (mod {P})\n"
    f"Top-{top_k} frequencies highlighted in red"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/phase1_fourier_spectrum.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Saved results/phase1_fourier_spectrum.png")
print(f"   Top {top_k} frequencies: {sorted(top_idx)}")
print(f"   (Compare with Phase 0 top freqs — same set → shared circuit; doubled set → split circuit)")

## Cell 10 — Head Ablation Sweep

Zero out each of the 4 attention heads one at a time, and measure:
- Accuracy **drop** on addition examples
- Accuracy **drop** on subtraction examples

**Interpretation guide:**
| Pattern | Interpretation |
|---------|---------------|
| Ablating head H drops **both** add and sub accuracy ≥10% | Head H is **shared** circuitry — used by both tasks |
| Ablating head H drops **only add** (sub stays >90%) | Head H is **add-specific** circuitry |
| Ablating head H drops **only sub** (add stays >90%) | Head H is **sub-specific** circuitry |
| Ablating head H barely affects either task | Head H is **not critical** for either task |

A head that hurts both tasks when ablated is the clearest evidence of shared computation.

**Saves to** `results/phase1_head_ablation.png`

In [ ]:
@torch.no_grad()
def ablate_head_accuracy(model, inputs, labels, op_tokens_map,
                          layer=0, head_idx=0, device=DEVICE):
    """
    Run the model with head `head_idx` in layer `layer` zeroed out.
    Returns per-task accuracy dict.

    How it works: we hook into the attention output *before* it's added to the
    residual stream, and zero out only the slice corresponding to head_idx.
    d_head = d_model // n_heads.
    """
    model.eval()
    inputs, labels = inputs.to(device), labels.to(device)
    d_head = D_MODEL // N_HEADS

    # Hook name for the per-head output (post-attention, pre-projection)
    # TransformerLens: 'blocks.{layer}.attn.hook_result'  shape [batch, seq, n_heads, d_head]
    hook_name = f"blocks.{layer}.attn.hook_result"

    def zero_head_hook(activation, hook):
        # activation: [batch, seq, n_heads, d_head]
        act = activation.clone()
        act[:, :, head_idx, :] = 0.0
        return act

    logits = model.run_with_hooks(
        inputs, fwd_hooks=[(hook_name, zero_head_hook)]
    )[:, -1, :]   # [N, vocab]

    preds = logits.argmax(-1)
    accs = {}
    for op_name, op_tok in op_tokens_map.items():
        mask = (inputs[:, 1] == op_tok)
        if mask.sum() == 0:
            accs[op_name] = float("nan")
        else:
            accs[op_name] = (preds[mask] == labels[mask]).float().mean().item()
    return accs


# ── Baseline (no ablation) ────────────────────────────────────────────────────
baseline_acc = per_task_accuracy(model, test_inputs, test_labels, OP_TOKENS)
print("Baseline test accuracies (no ablation):")
for op, acc in baseline_acc.items():
    print(f"  {op}: {acc:.4f}")

# ── Sweep over all heads ──────────────────────────────────────────────────────
ablation_results = {}   # head_idx -> {op_name: acc_drop}

for h in range(N_HEADS):
    ablated_acc = ablate_head_accuracy(
        model, test_inputs, test_labels, OP_TOKENS, layer=0, head_idx=h
    )
    ablation_results[h] = {
        op: baseline_acc[op] - ablated_acc[op]
        for op in OP_LIST
    }
    print(f"  Head {h}: acc drop — " +
          ", ".join(f"{op}: {ablation_results[h][op]:+.4f}" for op in OP_LIST))

print("\n✅ Head ablation sweep done.")

In [ ]:
# ── Plot: Head Ablation Bar Chart ─────────────────────────────────────────────
heads   = list(range(N_HEADS))
x       = np.arange(N_HEADS)
bar_w   = 0.35
COLORS  = {"add": "steelblue", "sub": "darkorange"}

fig, ax = plt.subplots(figsize=(10, 5))

for i, op in enumerate(OP_LIST):
    drops = [ablation_results[h][op] for h in heads]
    offset = (i - len(OP_LIST)/2 + 0.5) * bar_w
    bars = ax.bar(x + offset, drops, bar_w,
                  label=f"acc drop [{op}]",
                  color=COLORS.get(op, f"C{i}"),
                  edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, drops):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.axhline(0.10, color="red",  linestyle="--", linewidth=0.8, label="10% threshold")
ax.axhline(0.0,  color="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([f"Head {h}" for h in heads])
ax.set_ylabel("Accuracy drop when head is ablated (zeroed)")
ax.set_title(
    f"Phase 1 — Head Ablation Sweep (Layer 0, mod {P})\n"
    "Bars above the red dashed line = critical head for that task\n"
    "Both bars high → shared; only one bar high → task-specific"
)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
ax.set_ylim(-0.05, max(0.5, max(
    ablation_results[h][op] for h in heads for op in OP_LIST
) + 0.15))

plt.tight_layout()
plt.savefig("results/phase1_head_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved results/phase1_head_ablation.png")

# ── Interpretation helper ─────────────────────────────────────────────────────
THRESHOLD = 0.10
print("\n📊 Interpretation (threshold = 10% accuracy drop):")
for h in heads:
    critical_for = [op for op in OP_LIST if ablation_results[h][op] >= THRESHOLD]
    if len(critical_for) == len(OP_LIST):
        label = "🔗 SHARED — critical for ALL tasks"
    elif len(critical_for) == 1:
        label = f"🔀 TASK-SPECIFIC — critical only for [{critical_for[0]}]"
    elif len(critical_for) == 0:
        label = "💤 NOT CRITICAL for any task"
    else:
        label = f"🟡 PARTIAL — critical for {critical_for}"
    print(f"  Head {h}: {label}")

## Cell 11 — Activation Patching: Add ↔ Sub

**What it is:** We run the model on a **corrupted** input (e.g. a subtraction example), but mid-forward-pass we *replace* the activations at a specific hook with activations from a **clean** addition run. We then measure how much the output changes.

**What it tests:** If the model uses the *same* internal representation for add and sub, then patching add→sub should cause the output to shift towards the add answer (and vice versa). If they use *different* circuits, patching should have little effect.

**Hook points we test:**
1. `hook_embed` — token embedding output (tests whether representations overlap at input)
2. `blocks.0.attn.hook_result` — attention output (tests head-level sharing)
3. `blocks.0.hook_mlp_out` — MLP output (tests MLP sharing)
4. `hook_pos_embed` — positional embeddings (should be shared, a sanity check)

**Metric:** We report the **logit of the correct add answer** when patching add→sub. If this logit is high, the patched activations are driving the model towards the add answer, confirming shared representation.

In [ ]:
def get_per_task_examples(inputs, labels, op_tokens_map, n_examples=50):
    """Return (inputs, labels) subsets for each operation."""
    result = {}
    for op_name, op_tok in op_tokens_map.items():
        mask = (inputs[:, 1] == op_tok)
        idx  = mask.nonzero(as_tuple=True)[0][:n_examples]
        result[op_name] = (inputs[idx], labels[idx])
    return result


@torch.no_grad()
def activation_patch_accuracy(model, clean_inputs, corrupted_inputs,
                               corrupted_labels, hook_name, device=DEVICE):
    """
    Run corrupted_inputs through the model, patching `hook_name` activations
    with those from the clean run. Returns accuracy on corrupted_labels.

    If accuracy is HIGH: the patched hook activation carries the answer signal
       → that component is causally important and (at least partially) shared.
    If accuracy is LOW:  the hook doesn't carry the relevant signal.
    """
    model.eval()
    clean_inputs      = clean_inputs.to(device)
    corrupted_inputs  = corrupted_inputs.to(device)
    corrupted_labels  = corrupted_labels.to(device)

    # 1. Get clean activations
    _, clean_cache = model.run_with_cache(clean_inputs)
    clean_act      = clean_cache[hook_name]  # store the clean activation

    # 2. Run corrupted input, replacing the hook with the clean activation
    def patch_fn(activation, hook):
        # Broadcast if shapes differ (e.g. batch size mismatch)
        patched = activation.clone()
        batch   = min(patched.shape[0], clean_act.shape[0])
        patched[:batch] = clean_act[:batch]
        return patched

    patched_logits = model.run_with_hooks(
        corrupted_inputs, fwd_hooks=[(hook_name, patch_fn)]
    )[:, -1, :]   # [N, vocab]

    # Accuracy: does the patched model predict the CORRUPTED (original) labels correctly?
    # High accuracy here means the clean patch didn't overwrite the answer
    # → the patched component was not the critical one.
    # BUT: we also check whether the model now predicts the CLEAN labels
    #   (which would mean the patch injected the clean answer — evidence of sharing)
    corr_preds  = patched_logits.argmax(-1)
    corr_acc    = (corr_preds == corrupted_labels).float().mean().item()
    return corr_acc


# ── Build matched pairs ───────────────────────────────────────────────────────
# We pair add and sub examples with the SAME (a, b) values.
# This lets us ask: "if I patch add activations into a sub run for (5, 3),
#   does the model predict 5+3=8 instead of 5-3=2?"

N_PATCH = 113   # use all possible a values for a fixed b
FIXED_B = 7     # arbitrary, can sweep over several

# Construct (a, add_token, FIXED_B, =) for all a in [0, P)
add_tok = OP_TOKENS["add"]
sub_tok = OP_TOKENS["sub"]

add_inputs = torch.tensor(
    [[a, add_tok, FIXED_B, EQUALS_TOKEN] for a in range(P)], dtype=torch.long
).to(DEVICE)
sub_inputs = torch.tensor(
    [[a, sub_tok, FIXED_B, EQUALS_TOKEN] for a in range(P)], dtype=torch.long
).to(DEVICE)
add_labels = torch.tensor([(a + FIXED_B) % P for a in range(P)], dtype=torch.long).to(DEVICE)
sub_labels = torch.tensor([(a - FIXED_B) % P for a in range(P)], dtype=torch.long).to(DEVICE)

print(f"✅ Matched pairs built: {len(add_inputs)} examples (b={FIXED_B} fixed).")

# ── Baseline (no patching) ────────────────────────────────────────────────────
with torch.no_grad():
    add_logits = model(add_inputs)[:, -1, :]
    sub_logits = model(sub_inputs)[:, -1, :]
    add_base_acc = (add_logits.argmax(-1) == add_labels).float().mean().item()
    sub_base_acc = (sub_logits.argmax(-1) == sub_labels).float().mean().item()

print(f"Baseline acc (no patch): add={add_base_acc:.4f}, sub={sub_base_acc:.4f}")

In [ ]:
# ── Run patching at each hook point ──────────────────────────────────────────
HOOK_POINTS = [
    ("hook_embed",              "Token\nEmbed"),
    ("hook_pos_embed",          "Pos\nEmbed"),
    ("blocks.0.attn.hook_result", "Attn\nOutput"),
    ("blocks.0.hook_mlp_out",   "MLP\nOutput"),
]

patch_results = {}  # hook_key -> {"add→sub": acc, "sub→add": acc}

for hook_name, hook_label in HOOK_POINTS:
    # add→sub: patch add activations into sub run; measure sub accuracy
    # If sub_acc is LOW here, it means the add patch broke the sub answer
    # (the model was using that component's output for subtraction)
    try:
        a2s_acc = activation_patch_accuracy(
            model,
            clean_inputs=add_inputs,       # source of patched activations
            corrupted_inputs=sub_inputs,   # forward pass being patched
            corrupted_labels=sub_labels,   # measure accuracy for sub task
            hook_name=hook_name,
        )
        s2a_acc = activation_patch_accuracy(
            model,
            clean_inputs=sub_inputs,
            corrupted_inputs=add_inputs,
            corrupted_labels=add_labels,
            hook_name=hook_name,
        )
    except Exception as e:
        print(f"  ⚠️  Hook '{hook_name}' failed: {e}")
        a2s_acc, s2a_acc = float("nan"), float("nan")

    patch_results[hook_name] = {"add→sub": a2s_acc, "sub→add": s2a_acc, "label": hook_label}
    print(f"  [{hook_label.replace(chr(10), ' '):15s}]  add→sub acc: {a2s_acc:.4f}  |  sub→add acc: {s2a_acc:.4f}")

print("\n✅ Activation patching done.")

In [ ]:
# ── Plot: Activation Patching Results ─────────────────────────────────────────
hook_labels  = [v["label"] for v in patch_results.values()]
a2s_accs     = [v["add→sub"] for v in patch_results.values()]
s2a_accs     = [v["sub→add"] for v in patch_results.values()]

x       = np.arange(len(hook_labels))
bar_w   = 0.35

fig, ax = plt.subplots(figsize=(11, 5))

bars1 = ax.bar(x - bar_w/2, a2s_accs, bar_w, label="add→sub patch: sub accuracy",
               color="steelblue", edgecolor="white")
bars2 = ax.bar(x + bar_w/2, s2a_accs, bar_w, label="sub→add patch: add accuracy",
               color="darkorange", edgecolor="white")

# Baseline reference lines
ax.axhline(sub_base_acc, color="steelblue", linestyle="--", linewidth=0.8,
           label=f"Baseline sub acc ({sub_base_acc:.2f})")
ax.axhline(add_base_acc, color="darkorange", linestyle="--", linewidth=0.8,
           label=f"Baseline add acc ({add_base_acc:.2f})")

for bar, val in zip(list(bars1) + list(bars2), a2s_accs + s2a_accs):
    if not math.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(hook_labels)
ax.set_ylabel("Accuracy after patching")
ax.set_ylim(0, 1.15)
ax.set_title(
    f"Phase 1 — Activation Patching: Add ↔ Sub (mod {P}, b={FIXED_B})\n"
    "Low accuracy after patching → the patched component encodes operation-specific info\n"
    "Accuracy near baseline → the component is shared / not critical"
)
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/phase1_activation_patching.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved results/phase1_activation_patching.png")

# ── Interpretation guide ──────────────────────────────────────────────────────
print("\n📊 Quick interpretation:")
print("  If patching an add→sub causes sub acc to DROP significantly below baseline:")
print("    → that hook carries operation-specific information (disrupted by the patch)")
print("  If acc stays near baseline:")
print("    → the hook's activation is shared / the model ignores the patched value at that point")

## Cell 12 — Summary Dashboard

A clean summary of all Phase 1 findings. This is what you copy into your `docs/proposal.md` evidence log.

In [ ]:
print("=" * 72)
print("📊 PHASE 1 SUMMARY — Multi-Task Circuit Sharing: Add + Sub (mod 113)")
print("=" * 72)

print("\n1️⃣  GROKKING STATUS")
for op in OP_LIST:
    final = history[f"test_acc_{op}"][-1]
    status = "✅ GROKKED" if final >= 0.95 else "❌ DID NOT GROK"
    # Find grokking epoch
    grok_ep = next(
        (ep for ep, acc in zip(history["epoch"], history[f"test_acc_{op}"]) if acc > 0.95),
        "N/A"
    )
    print(f"   {op}: final test acc = {final:.4f}  {status}  (grok epoch ≈ {grok_ep})")

print("\n2️⃣  FOURIER ANALYSIS")
power = fourier_embedding_analysis(model, P)
top_freqs = sorted(np.argsort(power)[-6:][::-1].tolist())
print(f"   Top-6 dominant embedding frequencies: {top_freqs}")
print(f"   (Shared Fourier circuit → same ~4 freqs as Phase 0; split → double the freqs)")

print("\n3️⃣  HEAD ABLATION SWEEP")
THRESHOLD = 0.10
for h in range(N_HEADS):
    drops = {op: ablation_results[h][op] for op in OP_LIST}
    critical = [op for op, d in drops.items() if d >= THRESHOLD]
    if len(critical) == len(OP_LIST):
        label = "🔗 SHARED"
    elif len(critical) == 1:
        label = f"🔀 TASK-SPECIFIC [{critical[0]}]"
    elif len(critical) == 0:
        label = "💤 NOT CRITICAL"
    else:
        label = f"🟡 PARTIAL {critical}"
    drops_str = ", ".join(f"{op}: {d:.3f}" for op, d in drops.items())
    print(f"   Head {h}: {label}  (drops: {drops_str})")

print("\n4️⃣  ACTIVATION PATCHING (b=" + str(FIXED_B) + ")")
for hook_name, v in patch_results.items():
    lbl = v['label'].replace('\n', ' ')
    print(f"   {lbl:20s}: add→sub acc={v['add→sub']:.4f}  sub→add acc={v['sub→add']:.4f}")

print("\n5️⃣  SAVED RESULTS")
for fname in [
    "results/phase1_grokking_curves.png",
    "results/phase1_fourier_spectrum.png",
    "results/phase1_head_ablation.png",
    "results/phase1_activation_patching.png",
]:
    exists = "✅" if os.path.exists(fname) else "❌"
    print(f"   {exists} {fname}")

print("\n" + "=" * 72)
print("Next step: commit results, update docs/proposal.md, then start Phase 2 (add multiplication).")
print("=" * 72)

## Cell 13 — (Optional) Save Checkpoint & Mount Google Drive

If you want to keep your trained model weights between Colab sessions, mount Drive and save there.
Skip this cell if you'll re-train from scratch each time (only ~15 min on T4).

In [ ]:
# ── Optional: Mount Google Drive ──────────────────────────────────────────────
SAVE_TO_DRIVE = False   # set True if you want to persist weights

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = "/content/drive/MyDrive/mech_interp_phase1"
    os.makedirs(save_dir, exist_ok=True)

    # Save final model weights
    torch.save(model.state_dict(), f"{save_dir}/model_final.pt")
    print(f"✅ Model saved to {save_dir}/model_final.pt")

    # Save history
    import json
    with open(f"{save_dir}/history.json", "w") as f:
        json.dump(history, f)
    print(f"✅ Training history saved.")
else:
    print("ℹ️  Drive save skipped (SAVE_TO_DRIVE=False). Set True and re-run if needed.")